# Three-phase EMT RL-FMU stability benchmark

This notebook presents the measured native, FMI Model Exchange (ME), single-pass explicit FMI Co-Simulation, and rollback/Newton Co-Simulation results for the grounded-wye ABC RL circuit. Run `python3 dpsim-fmi/examples/linear_rl_stability_3ph/run_benchmark.py` from the repository root before executing the result cells.

## Explicit settings

| Setting | Value | Consistency |
|---|---|---|
| Source | Balanced 50 Hz, 10 V RMS phase-to-neutral; 1 ms amplitude ramp | Same waveform and phase order in all variants |
| Grid resistance | 1 Ω per phase | Outside the FMU in ME/CS; same native resistor |
| FMU resistance | 0.1 Ω per phase | Same branch equation and native resistor |
| Inductance | 20 µH per phase | Same branch state equation |
| Initialization | `i_abc(0)=[0,0,0] A`; `u_abc(0)=[0,0,0] V` | Same state and terminal voltage in native, ME, and CS; no load-flow initialization |
| Nominal step | 50 µs network and CS communication step | One `doStep()` each network interval |
| Stop time | 20 ms (one full 50 Hz cycle) | Same duration in all variants |
| Explicit CS | Held PCC voltage and last completed FMU current; one `doStep()` per interval | No iteration or rollback |
| Compensated CS | Rollback/Newton at 50 µs | Save/restore FMU state and replay interval while solving interface voltage/current |
| Sweep | 20, 30, 35, 38, 39, 40, 40.13, 40.25, 41, 45, 50 µs | Same sweep for native, ME, and CS |

All branch currents and terminal voltages start at zero in every variant. The balanced source amplitude ramps linearly to its 10 V RMS value over 1 ms, so the source is also initialized consistently.


## Why the stability boundary is unchanged

For each independent phase, the physical branch is `L di/dt = u - Rf i`; the grid solves `u_n = v_s,n - Rg i_n` from the last accepted current. The FMU's exact held-input update is `i_(n+1) = a i_n + (1-a)u_n/Rf`, where `a=exp(-Rf h/L)`. Substitution gives the homogeneous error map `e_(n+1)=q(h)e_n`, with

`q(h) = (1 + Rg/Rf) exp(-Rf h/L) - Rg/Rf`.

A time-varying balanced AC source changes the forcing term only. The same q multiplies the error independently in a, b, and c. For `Rg=1 Ω`, `Rf=0.1 Ω`, and `L=20 µH`, `q=-1` at `h=40.134139 µs`; above that, the single-pass sampled error alternates and grows. This instability belongs to the explicit policy, not to the passive RL plant or to every FMI CS master. A master that restores the FMU state and iterates to a consistent interface point removes the one-step feedback lag for this case.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

results = Path("build-fmi/linear_rl_stability_3ph/results")
summary = pd.read_csv(results / "summary.csv")
metrics = json.loads((results / "metrics.json").read_text())
print(f"Analytical boundary: {metrics['analytical_cs_stability_limit_us']:.6f} µs")
summary[summary.step_us.eq(50)][
    [
        "mode",
        "cs_multiplier_per_phase",
        "peak_abs_phase_current_A",
        "max_abs_delta_to_native_A",
        "rms_delta_to_native_A",
        "final_i_a_A",
        "final_i_b_A",
        "final_i_c_A",
    ]
]

In [ ]:
cs = summary[summary["mode"].eq("cs")][
    [
        "step_us",
        "cs_multiplier_per_phase",
        "cs_stable_by_analysis",
        "peak_abs_phase_current_A",
    ]
]
cs

In [ ]:
display(Image(filename=str(results / "linear_rl_3ph_stability.png")))

## Reading the measurements

## Comparison at the same 50 µs macro-step

Native DPsim and ME match exactly at logged precision (max ABC current difference 0 A). Rollback/Newton CS stays bounded with a 12.856253 A peak, nearly the native/ME 12.856276 A peak. Its pointwise max phase-current difference from native is 0.670 A and its RMS difference is 0.172 A over the 20 ms cycle. The one-pass CS run reaches about `7.61×10⁶¹ A`. It has the same FMU and macro-step, so the difference is the coupling algorithm.

For rollback/Newton, DPsim checkpoints the FMU, guesses endpoint voltage, restores the checkpoint and replays the interval. During replay it samples a linear voltage trajectory at four substep midpoints; each sample is held over a 12.5 µs `doStep()`. It then checks the network/FMU interface residual. In this linear example it converged in one Newton iteration per communication interval (maximum logged residual about 3×10⁻⁶ A). Replay and iteration restore a self-consistent interface solution.

Below 40.134139 µs the single-pass method is linearly stable, though convergence becomes slow near the threshold. Reducing the communication step is the simplest compensation. Rollback/Newton allows this 50 µs example to remain bounded but costs checkpoint/replay work and requires an FMU that supports state save/restore.
